In [10]:
import os
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split

from torch.optim.lr_scheduler import CosineAnnealingLR

from torch.amp import autocast, GradScaler

from sklearn.model_selection import StratifiedKFold

In [11]:
data_path = '/kaggle/input/competitions/cassava-leaf-disease-classification'

df = pd.read_csv(os.path.join(data_path, 'train.csv'))
df.head()

,image_id,label
0,1000015157.jpg,0
1,1000201771.jpg,3
2,100042118.jpg,1
3,1000723321.jpg,1
4,1000812911.jpg,3


In [12]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.RandomResizedCrop(size=(288, 288), scale=(0.8, 1.0), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.1,
        rotate_limit=15,
        p=0.5
    ),
    A.HueSaturationValue(
        hue_shift_limit=10,
        sat_shift_limit=10,
        val_shift_limit=10,
        p=0.3
    ),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

valid_transform = A.Compose([
    A.Resize(288, 288),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [13]:
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
import os

class CassavaDataset(Dataset):
    def __init__(self, df, data_path, transform=None):
        self.df = df
        self.data_path = data_path
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.data_path, self.df.iloc[idx]['image_id'])
        image = np.array(Image.open(img_path).convert("RGB"))
        label = self.df.iloc[idx]['label']

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]

        return image, label

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 5)
model = model.to(device)

cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 193MB/s]


In [15]:
scaler = GradScaler('cuda')

In [16]:
num_epochs = 30

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.Adam(model.parameters(), lr=3e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)

In [17]:
import numpy as np
import torch

def mixup_data(x, y, alpha=0.4):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [18]:
from torch.amp import autocast

def train_one_epoch(model, loader, criterion, optimizer, device, scaler, alpha=0.4):
    model.train()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Mixup 적용
        mixed_images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=alpha)

        with autocast(device_type='cuda'):
            outputs = model(mixed_images)
            loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        # acc 계산은 보통 원래 라벨 기준으로 근사 계산
        _, preds = torch.max(outputs, 1)
        correct += (lam * (preds == labels_a).sum().item() +
                    (1 - lam) * (preds == labels_b).sum().item())
        total += labels.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc

In [19]:
from torch.amp import autocast

def valid_one_epoch(model, loader, criterion, device):
    model.eval()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            # AMP 적용 
            with autocast(device_type='cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc

In [20]:
import torch.nn.functional as F

def valid_one_epoch_tta(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits1 = model(images)
            logits2 = model(torch.flip(images, dims=[3]))

            probs1 = F.softmax(logits1, dim=1)
            probs2 = F.softmax(logits2, dim=1)

            probs = (probs1 + probs2) / 2
            preds = probs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [21]:
num_epochs = 30
patience = 7
n_splits = 5

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(df, df['label'])):
    print(f"\n========== Fold {fold+1}/{n_splits} ==========")

    fold_train_df = df.iloc[train_idx].reset_index(drop=True)
    fold_valid_df = df.iloc[valid_idx].reset_index(drop=True)

    train_dataset = CassavaDataset(fold_train_df, train_img_dir, transform=train_transform)
    valid_dataset = CassavaDataset(fold_valid_df, train_img_dir, transform=valid_transform)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, num_workers=2)

    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 5)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.Adam(model.parameters(), lr=3e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
    scaler = GradScaler('cuda')

    best_acc = 0
    counter = 0
    best_model_path = f"best_model_fold{fold+1}.pth"

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device, scaler, alpha=0.2
        )
        valid_loss, valid_acc = valid_one_epoch(
            model, valid_loader, criterion, device
        )

        print(f"Fold {fold+1} Epoch [{epoch+1}/{num_epochs}]")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Valid Loss: {valid_loss:.4f} | Valid Acc: {valid_acc:.4f}")
        print("-" * 40)

        scheduler.step()

        if valid_acc > best_acc:
            best_acc = valid_acc
            counter = 0
            torch.save(model.state_dict(), best_model_path)
        else:
            counter += 1

        if counter >= patience:
            print("Early Stopping 발생")
            break

    model.load_state_dict(torch.load(best_model_path, map_location=device))
    tta_acc = valid_one_epoch_tta(model, valid_loader, device)
    print(f"Fold {fold+1} TTA Valid Acc: {tta_acc:.4f}")

    fold_scores.append(tta_acc)

print("\n========== K-Fold Result ==========")
for i, score in enumerate(fold_scores):
    print(f"Fold {i+1}: {score:.4f}")

print(f"Mean TTA Valid Acc: {sum(fold_scores)/len(fold_scores):.4f}")


========== Fold 1/5 ==========
Fold 1 Epoch [1/30]
Train Loss: 0.8535 | Train Acc: 0.7415
Valid Loss: 0.6736 | Valid Acc: 0.8152
----------------------------------------
Fold 1 Epoch [2/30]
Train Loss: 0.7541 | Train Acc: 0.7840
Valid Loss: 0.6843 | Valid Acc: 0.8026
----------------------------------------
Fold 1 Epoch [3/30]
Train Loss: 0.7376 | Train Acc: 0.7902
Valid Loss: 0.6106 | Valid Acc: 0.8346
----------------------------------------
Fold 1 Epoch [4/30]
Train Loss: 0.7060 | Train Acc: 0.8027
Valid Loss: 0.6266 | Valid Acc: 0.8360
----------------------------------------
Fold 1 Epoch [5/30]
Train Loss: 0.6978 | Train Acc: 0.8096
Valid Loss: 0.6088 | Valid Acc: 0.8367
----------------------------------------
Fold 1 Epoch [6/30]
Train Loss: 0.7067 | Train Acc: 0.8052
Valid Loss: 0.5924 | Valid Acc: 0.8495
----------------------------------------
Fold 1 Epoch [7/30]
Train Loss: 0.6692 | Train Acc: 0.8230
Valid Loss: 0.5879 | Valid Acc: 0.8463
------------------------------------

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

def valid_one_epoch_ensemble_tta(model_paths, loader, device, num_classes=5):
    models_list = []

    # 1) fold별 저장 모델 불러오기
    for path in model_paths:
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        model.load_state_dict(torch.load(path, map_location=device))
        model = model.to(device)
        model.eval()
        models_list.append(model)

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            ensemble_probs = 0

            # 2) 각 모델 예측 + TTA
            for model in models_list:
                logits1 = model(images)                          # 원본
                logits2 = model(torch.flip(images, dims=[3]))   # 좌우반전

                probs1 = F.softmax(logits1, dim=1)
                probs2 = F.softmax(logits2, dim=1)

                probs = (probs1 + probs2) / 2
                ensemble_probs += probs

            # 3) fold 모델 평균
            ensemble_probs = ensemble_probs / len(models_list)

            preds = ensemble_probs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [23]:
model_paths = [
    "best_model_fold1.pth",
    "best_model_fold2.pth",
    "best_model_fold3.pth",
    "best_model_fold4.pth",
    "best_model_fold5.pth"
]

ensemble_tta_acc = valid_one_epoch_ensemble_tta(model_paths, valid_loader, device)
print(f"Ensemble TTA Valid Acc: {ensemble_tta_acc:.4f}")

Ensemble TTA Valid Acc: 0.9717
